# <b>Preprocessing count matrices with scDblFinder</b>
### scDblFinder is used to find and remove doublets
### This notebook was inspired by: https://github.com/mousepixels/sanbomics_scripts/blob/main/soupX/soupX_python_test.ipynb

In [ ]:
### Import packages using the scDblFinderEnv environment
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import median_abs_deviation as mad

sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)

In [ ]:
### List of the folder names that contain the filtered feature matrices.
sample_ids = ['Kidney_1','Kidney_2','Kidney_3', 'Mes_1', 'Mes_2', 'Mes_3', 'TA_1','TA_2', 'TA_3']

In [ ]:
### Preprocessing samples with scanpy.
def is_outlier(adata, metric, nmads):
    M = adata.obs[metric]
    return (M < np.median(M) - nmads * mad(M)) | (M > np.median(M) + nmads * mad(M))

def pp(sample_id):
    print(sample_id)
    ### Build the adata object
    adata = sc.read_mtx('../data/' + sample_id + '/DGE_filtered/count_matrix.mtx.gz')
    adata.obs = pd.read_csv('../data/' + sample_id + "/DGE_filtered/cell_metadata.csv.gz", index_col=None).set_index('bc_wells')
    #adata.obs.index = adata.obs.bc_wells
    adata.var = pd.read_csv('../data/' + sample_id + "/DGE_filtered/all_genes.csv.gz", index_col = None).set_index('gene_name')
    #adata.var.index = adata.var.gene_name 
    adata.var_names_make_unique()
    adata.obs['sample_id'] = sample_id
    adata.obs['platform'] = 'Parse'
       
    
    ### calculate QC metrics
    # mitochondrial genes
    adata.var["mt"] = adata.var_names.str.startswith("Mt-")
    # ribosomal genes
    adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))
    # hemoglobin genes.
    adata.var["hb"] = adata.var_names.str.contains(("^Hb[^(P)]"))

    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], 
                               inplace=True, percent_top=[20], log1p=True)
    
    ### Plot QC metrics    
    p1 = sns.displot(adata.obs["total_counts"], bins=100, kde=False)
    p2 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
    p3 = sc.pl.violin(adata, ['log1p_n_genes_by_counts', 'log1p_total_counts', "pct_counts_mt", 'pct_counts_ribo'],
             jitter=0.4, multi_panel=True)
    
    ### filter outliers
    adata.obs["outlier"] = (
        is_outlier(adata, "log1p_total_counts", 5)
        | is_outlier(adata, "log1p_n_genes_by_counts", 5)
        | is_outlier(adata, "pct_counts_in_top_20_genes", 5)
    )

    adata.obs["mt_outlier"] = (adata.obs["pct_counts_mt"] > 2)
    adata.obs["ribo_outlier"] = (adata.obs["pct_counts_ribo"] > 2)


    print(adata.obs.outlier.value_counts())
    print(adata.obs.mt_outlier.value_counts())
    print(adata.obs.ribo_outlier.value_counts())
    print(f"Total number of cells: {adata.n_obs}")
    adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier) & (~adata.obs.ribo_outlier)].copy()
    
    print(f"Number of cells after filtering of low quality cells: {adata.n_obs}")
    p1 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
    p2 = sc.pl.violin(adata, ['log1p_n_genes_by_counts', 'log1p_total_counts', "pct_counts_mt", 'pct_counts_ribo'],
             jitter=0.4, multi_panel=True)
    
    return adata

In [ ]:
adatas = [pp(adata) for adata in sample_ids]

In [ ]:
### Apply minimum cell and gene filters
for adata in adatas:
    print(f"Total number of genes: {adata.n_vars}")
    # Min 1 cells - filters out 0 count genes
    sc.pp.filter_genes(adata, min_cells=3)
    print(f"Number of genes after cell filter: {adata.n_vars}")

    ## Filter cells
    print(f"Total number of cells: {adata.n_obs}")
    sc.pp.filter_cells(adata, min_genes = 100)
    print(f"Number of cells after gene filter: {adata.n_obs}")

In [ ]:
## Run this if you want to save the data before moving on to doublet detection
# i = 0
# for sample_name in sample_ids:
#     adatas[i].write("../data/" + sample_name + "/" + sample_name + "_raw.h5ad")
#     i+=1

## Doublet Detection

In [ ]:
### This is necessary to run R code in Python
import anndata2ri
import logging

import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [ ]:
%%R
library(Seurat)
library(scater)
library(scDblFinder)
library(BiocParallel)

dbl_finder <- function(data_mat){
    set.seed(123)
    sce = scDblFinder(
        SingleCellExperiment(
            list(counts=data_mat),
        ) 
    )
    doublet_score = sce$scDblFinder.score
    doublet_class = sce$scDblFinder.class
    return (c(doublet_score, doublet_class))
}

In [ ]:
def doublet_removal(adata):
    
    data_mat = adata.X.T

    # Execute the R code and get the corrected counts
    %R -i data_mat -o out out = dbl_finder(data_mat)

    half = int(0.5 * len(out))
    doublet_score = out[:half]
    doublet_class = out[half:]
    doublet_class = doublet_class.astype(str)
    doublet_class[doublet_class == '1.0'] = 'singlet'
    doublet_class[doublet_class == '2.0'] = 'doublet'
    
    ### Creates two columns in adata.obs to store scDblFinder data
    adata.obs["scDblFinder_score"] = doublet_score
    adata.obs["scDblFinder_class"] = doublet_class
    adata.obs.scDblFinder_class.value_counts()
    
    return adata

In [ ]:
### Runs scDblFinder. Note that this only labels cells as singlets or doublets, it does not automatically remove them from the adata objects.
adatas = [doublet_removal(adata) for adata in adatas]

In [ ]:
### Save the soupx/scdblfinder preprocessed data

i = 0
for sample_id in sample_ids:
    adatas[i].write("../data/" + sample_id + "_preprocessed.h5ad")
    i+=1